**Icha Meisya Kurnia (25031554047)**

**Putri Najwa Zuleyka Al Hasni (25031554076)**

In [ ]:
import pandas as pd
import requests
import warnings
warnings.filterwarnings('ignore')

url = 'https://api.spaceflightnewsapi.net/v4/articles/'
response = requests.get(url)
data = response.json()

df = pd.DataFrame(data['results'])

df_pre = df['summary'].dropna().reset_index(drop=True)
print(f"Total data artikel: {len(df_pre)}")


Total data artikel: 10


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
from nltk.tokenize import word_tokenize
from tqdm import tqdm

sentences = [word_tokenize(text.lower()) for text in tqdm(df_pre)]

100%|██████████| 10/10 [00:00<00:00, 1622.62it/s]


In [ ]:
!pip install gensim umap-learn
import gensim.models

model = gensim.models.Word2Vec(
    sentences,
    min_count=1,
    vector_size=100,
    window=5,
    sg=0
)

model.save('space_news.w2v')
model = gensim.models.Word2Vec.load('space_news.w2v')

w2v = model.wv
print("Word2Vec Info")
print("Top 10 Words:", w2v.index_to_key[:10])
print("Vector Shape:", w2v.vectors.shape)

sample_word = w2v.index_to_key[0]
print(f"Similar words to '{sample_word}':", w2v.similar_by_key(sample_word))

Word2Vec Info
Top 10 Words: ['the', ',', 'a', '.', 'and', 's', '’', 'apod', 'nasa', 'of']
Vector Shape: (289, 100)
Similar words to 'the': [('education', 0.30167728662490845), ('temperature', 0.27882492542266846), ('earth', 0.2760356664657593), ('should', 0.23781462013721466), ('space', 0.22643287479877472), ('pyrocb', 0.21842163801193237), ('of', 0.21702677011489868), ('chamber', 0.2166740894317627), ('!', 0.20772141218185425), ('at', 0.19768503308296204)]


In [ ]:
import numpy as np

def get_document_vector(tokens, model):
    valid_vectors = [model.wv[word] for word in tokens if word in model.wv]
    if not valid_vectors:
        return np.zeros(model.vector_size)
    return np.mean(valid_vectors, axis=0)

X_features_w2v = np.array([get_document_vector(doc, model) for doc in sentences])
df_w2v = pd.DataFrame(X_features_w2v)
df_w2v.head()

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-2.610645e-05,0.000339,0.000377,0.000808,-0.000274,-0.001984,0.001294,0.002147,-0.001636,-0.001870,...,0.001728,0.000271,-0.000781,0.000572,0.002070,0.000456,0.000841,-0.001055,0.001351,-0.000824
1,-8.639179e-04,0.000581,0.001365,-0.000075,0.000954,-0.000300,0.001294,0.001714,-0.000029,0.000808,...,0.002142,0.000562,0.000758,0.001176,0.001308,0.002028,0.000102,0.000753,-0.000962,-0.001096
2,-3.234794e-07,0.000387,0.000406,0.000673,-0.000307,-0.001725,0.001032,0.002088,-0.001668,-0.001775,...,0.001848,0.000346,-0.000609,0.000760,0.001993,0.000043,0.000327,-0.001491,0.001523,-0.000268
3,-5.498544e-04,0.002163,-0.000277,-0.000228,0.001159,-0.000459,0.001709,0.001886,-0.000926,-0.000867,...,0.000903,0.001695,0.000311,-0.000816,0.000953,0.001404,0.000670,-0.000790,0.000528,-0.000441
4,-9.312952e-04,0.000852,0.000368,0.000736,0.000719,-0.001006,-0.000472,0.000821,-0.000558,-0.001176,...,0.001456,0.000234,-0.000247,-0.000173,0.001260,0.000743,0.000125,0.000560,-0.000938,0.000267


In [ ]:
from umap import UMAP
import plotly.express as px

X_umap = UMAP().fit_transform(w2v.vectors)
df2_w2v = pd.DataFrame(X_umap, columns=['umap1', 'umap2'])
df2_w2v['word'] = w2v.index_to_key

fig_w2v = px.scatter(df2_w2v, x='umap1', y='umap2', text='word', title='Word2Vec Visualization')
fig_w2v.update_traces(textposition='top center')
fig_w2v.update_layout(height=800)
fig_w2v.show()

In [ ]:
model_fasttext = gensim.models.FastText(
    sentences,
    min_count=1,
    vector_size=100,
    window=5,
    min_n=3,
    max_n=6,
    sg=1,
    epochs=10
)
model_fasttext.save('space_news.fasttext')
model_fasttext = gensim.models.Word2Vec.load('space_news.fasttext')

fasttext = model_fasttext.wv
print("FastText Info")
print("Top 10 Words:", fasttext.index_to_key[:10])
print("Vector Shape:", fasttext.vectors.shape)
print(f"Similar words to '{sample_word}':", fasttext.similar_by_key(sample_word))

X_features_ft = np.array([get_document_vector(doc, model_fasttext) for doc in sentences])
df_ft = pd.DataFrame(X_features_ft)

X_umap_ft = UMAP().fit_transform(fasttext.vectors)
df2_ft = pd.DataFrame(X_umap_ft, columns=['umap1', 'umap2'])
df2_ft['word'] = fasttext.index_to_key

fig_ft = px.scatter(df2_ft, x='umap1', y='umap2', text='word', title='FastText Visualization')
fig_ft.update_traces(textposition='top center')
fig_ft.update_layout(height=800)
fig_ft.show()

FastText Info
Top 10 Words: ['the', ',', 'a', '.', 'and', 's', '’', 'apod', 'nasa', 'of']
Vector Shape: (289, 100)
Similar words to 'the': [('international', 0.9870866537094116), ('station', 0.9869687557220459), ('exploration', 0.9865624308586121), ('investigation', 0.9864193797111511), ('intentions', 0.9864007234573364), ('inspiration', 0.9863462448120117), ('explanation', 0.9863226413726807), ('stations', 0.9860455393791199), ('evaluation', 0.9858590960502625), ('innovation', 0.9855125546455383)]
